# 0.28 — `detect_theme`: run the funnel on any theme / date

Thin demo of the reusable wrapper in **`code/theme_detect.py`**. Feed a **corpus of headlines**
plus an **important date**, get back the tidy per-`(week, anchor)` dataframe with every funnel step:

`candidates → caught → promoted → llm_kept`  (+ `rank_candidate / rank_caught / rank_promoted / llm_judged`)

Every threshold is a keyword argument; the LLM gate is **off by default**.

**Corpus**: the single preprocessed file `output/news_corpus.parquet` (ADD-event dating, 2010–2025 —
built by `scripts/preprocess_news.py`; replay artifact proven in 0.25, layout in 0.26). The first cell
is a **gate**: it checks the corpus is built and up to date, and builds it from raw only if needed.
Never read `output/tmp/terms_{year}.parquet` in analyses — those are disposable build caches.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))                    # add code/ to the path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import pandas as pd
import polars as pl
from theme_detect import detect_theme
import preprocess_news as pp

CORPUS = pp.ensure()               # gate: builds missing years / re-merges ONLY if needed (no-op when certified)

def load_corpus(y0, y1):
    """[Headline, date, terms] slice of THE corpus, calendar years y0..y1 (lazy — never loads all years)."""
    return (pl.scan_parquet(CORPUS)
              .filter(pl.col("date").dt.year().is_between(y0, y1))
              .collect().to_pandas())

pd.set_option("display.max_colwidth", 46); pd.set_option("display.width", 200)
print("imported detect_theme from", detect_theme.__module__)

2010: certified add-event-v1 build exists — skipping
2011: certified add-event-v1 build exists — skipping
2012: certified add-event-v1 build exists — skipping
2013: certified add-event-v1 build exists — skipping
2014: certified add-event-v1 build exists — skipping
2015: certified add-event-v1 build exists — skipping
2016: certified add-event-v1 build exists — skipping
2017: certified add-event-v1 build exists — skipping
2018: certified add-event-v1 build exists — skipping
2019: certified add-event-v1 build exists — skipping
2020: certified add-event-v1 build exists — skipping
2021: certified add-event-v1 build exists — skipping
2022: certified add-event-v1 build exists — skipping
2023: certified add-event-v1 build exists — skipping
2024: certified add-event-v1 build exists — skipping
2025: certified add-event-v1 build exists — skipping
corpus up to date: news_corpus.parquet
imported detect_theme from theme_detect


## 1 · Quantum around the CHPX ETF date (2025-01-15)

Slice the corpus (any `[Headline, date, terms]` frame works; raw `[Headline, date]` is
extracted internally) and pass a quantum `theme_hint` to flag relevant anchors.

In [2]:
QUANTUM = r"\bionq\b|rigetti|\bd-wave\b|\bdwave\b|quantinuum|\bqubit|quantum comput|psiquantum|quantum advantage"
corpus = load_corpus(2022, 2025)

df = detect_theme(corpus, "2025-01-15", theme_hint=QUANTUM)      # LLM off by default
print(f"{len(df):,} rows · {int(df.caught.sum()):,} caught · {int(df.promoted.sum())} promoted · {int(df.is_theme.sum())} theme row(s)")
df[df.is_theme]

5,104 rows · 3,789 caught · 41 promoted · 1 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
4875,2025-01-07/2025-01-13,ionq delta,"daily, daily volume, delta, hedge, ionq, o...",3,9,1.0,True,True,False,<NA>,IonQ Delta Hedge at 12% Daily Volume: Opti...,107,151,<NA>,False


In [3]:
df[df.promoted].sort_values("clustering")

,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
3670,2024-12-03/2024-12-09,hegseth,"trump, ernst, cbs, defense, fighting, hear...",9,34,0.343,False,True,True,<NA>,*HEGSETH SAYS HE HEARD FROM TRUMP TODAY TO...,5,4,4,True
3978,2024-12-10/2024-12-16,mangione,"luigi, luigi mangione, murder, suspect, su...",6,30,0.371,False,True,True,<NA>,*PROSECUTORS FILED MURDER & OTHER CHARGES ...,19,8,8,True
2374,2024-10-29/2024-11-04,altair engineering,"altair, engineering, bausch, lomb, survey,...",5,18,0.410,False,True,True,<NA>,"Altair Engineering, Bausch + Lomb Are Top ...",17,12,34,False
3437,2024-11-26/2024-12-02,apotea,"price, final, sek58, set, share, apotea fi...",7,25,0.429,False,True,True,<NA>,*APOTEA FINAL PRICE IN OFFERING IS SET TO ...,4,8,19,True
3155,2024-11-19/2024-11-25,groupe dynamite,"dynamite, groupe, canadian, fall, ahead, g...",9,23,0.429,False,True,True,<NA>,*GROUPE DYNAMITE IPO GOES AHEAD AT C$21/SH...,4,13,24,True
3145,2024-11-19/2024-11-25,lucid capital,"capital, lucid, investment, credit, atyr, ...",13,34,0.429,False,True,True,<NA>,Atyr Pharma Rated New Buy at Lucid Capital...,2,4,5,True
2454,2024-10-29/2024-11-04,fnz,"ahlsell, holding, parts, parts holding, ta...",3,11,0.436,False,True,True,<NA>,"Parts Holding, TAP, Ahlsell, FNZ",86,83,41,False
3190,2024-11-19/2024-11-25,mokingran,"hong, kong, hong kong, mokingran offers, o...",5,15,0.438,False,True,True,<NA>,*MOKINGRAN OFFERS ABOUT 44M SHARES IN HONG...,25,43,37,False
5004,2025-01-14/2025-01-20,loulo-gounkoto,"barrick, mali, barrick provides, loulo-gou...",3,16,0.457,False,True,True,<NA>,*BARRICK PROVIDES FURTHER UPDATE ON LOULO-...,42,14,36,False
1870,2024-10-15/2024-10-21,opella,"sanofi, talks, transfer, bpifrance, france...",19,84,0.467,False,True,True,<NA>,*PAI PARTNERS IS SAID TO SUBMIT REVISED BI...,1,1,2,True


## 2 · A different theme — genAI around the CHAT ETF (2023-05-18)

Swap the regex and the date; nothing else changes. Here we widen `detect_months` to **9** so the window
reaches back before ChatGPT's Nov-2022 birth (with the default 5-month window it is already in the
baseline, hence not *novel*) — a good illustration of a tunable knob.

In [ ]:
GENAI = r"chatgpt|generative ai|\bgenai\b|large language model"
corpus_ai = load_corpus(2020, 2023)

ai = detect_theme(corpus_ai, "2023-05-18", theme_hint=GENAI, detect_months=9)
print(f"{len(ai):,} rows · {int(ai.caught.sum()):,} caught · {int(ai.promoted.sum())} promoted · {int(ai.is_theme.sum())} theme row(s)")
ai[ai.is_theme].head(20)                                          # chatgpt: caught -> promoted

## 3 · Tuning knobs

Every threshold is a plain keyword argument — loosen the gates inline to catch smaller / faster signals.

In [ ]:
loose = detect_theme(corpus, "2025-01-15", theme_hint=QUANTUM,
                     degree_min=6, cluster_max=0.7, persist_weeks=1)
print(f"looser gates: {len(loose):,} rows · {int(loose.caught.sum()):,} caught · {int(loose.promoted.sum())} promoted")
loose[loose.is_theme]

looser gates: 5,678 rows · 5,155 caught · 1239 promoted


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
5407,2025-01-07/2025-01-13,ionq delta,"daily, daily volume, delta, hedge, ionq, o...",3,9,1.0,True,True,False,<NA>,IonQ Delta Hedge at 12% Daily Volume: Opti...,139,205,<NA>,False


## 4 · Theme birth — crypto around the first bitcoin mania (2014-01-15)

The first bitcoin bubble: Silk Road shutdown **2013-10-02**, US Senate hearing **2013-11-18**,
bitcoin > $1,000 **2013-11-27**, China PBOC ban **2013-12-05**, Dogecoin born **2013-12-06**.
Headline flow: ~12/mo (Jan–Aug 2013) → 74 → 157 → 201/mo (Nov 2013–Jan 2014), a ~7× burst.
Investability markers *predate* the mania — Winklevoss ETF S-1 **2013-07-01**, GBTC launch
**2013-09-25** — the same product-led pattern as quantum.

Detection window (5 mo before `2014-01-15`) covers the whole wave; the 24-month baseline
(Aug 2011 – Aug 2013) now exists thanks to the 2010+ corpus and *contains the 2011 and Apr-2013
mini-bubbles* — so `bitcoin` itself is baseline vocabulary and can never be an anchor. The funnel
must find the wave through the vocabulary it minted: `litecoin`, `dogecoin`, new bigrams of old
words. That is the point of this test.

In [ ]:
CRYPTO = r"bitcoin|litecoin|dogecoin|\bbtc\b|\bcrypto|virtual currenc|digital currenc|mt gox|\bmtgox\b|coinbase"
corpus_btc = load_corpus(2011, 2014)

btc = detect_theme(corpus_btc, "2014-01-15", theme_hint=CRYPTO)
print(f"{len(btc):,} rows · {int(btc.caught.sum()):,} caught · {int(btc.promoted.sum())} promoted · {int(btc.is_theme.sum())} theme row(s)")
btc[btc.is_theme].head(20)